# LangChain 核心模块学习：Model I/O

`Model I/O` 是 LangChain 为开发者提供的一套面向 LLM 的标准化模型接口，包括模型输入（Prompts）、模型输出（Output Parsers）和模型本身（Models）。

- Prompts：模板化、动态选择和管理模型输入
- Models：以通用接口调用语言模型
- Output Parser：从模型输出中提取信息，并规范化内容

不要混用新旧版本：LangChain 1.0 是一次不兼容更新，langchain-core 从 0.x 升到 1.x 后，旧的 langchain-community 等包必须同步升级，否则必然冲突。
优先用虚拟环境：LangChain 依赖复杂，不同项目的版本要求差异大，用虚拟环境可以避免互相干扰。

## 模型抽象 Model

- 语言模型(LLMs): LangChain 的核心组件。LangChain并不提供自己的LLMs，而是为与许多不同的LLMs（OpenAI、Cohere、Hugging Face等）进行交互提供了一个标准接口。
    OpenAI Completion API（旧版）
    文本补全模型
- 聊天模型(Chat Models): 语言模型的一种变体。虽然聊天模型在内部使用了语言模型，但它们提供的接口略有不同。与其暴露一个“输入文本，输出文本”的API不同，它们提供了一个以“聊天消息”作为输入和输出的接口。

（注：对比 OpenAI Completion API和 Chat Completion API）
```
BaseLanguageModel  # 顶层祖宗类：所有模型都继承它
├── BaseLLM         # 文本补全模型基类
│     └── LLM
│         └── 各种文本模型（OpenAI/智谱/通义）
└── BaseChatModel   # 聊天模型基类
      └── ChatOpenAI、ChatZhipuAI、ChatTongyi
```

## 聊天模型（Chat Models)

输入：一堆消息（系统提示 + 用户说的 + AI 说的）
输出：AI 回复消息
专门用于对话、多轮聊天、工具调用


类继承关系：

```
BaseLanguageModel --> BaseChatModel --> <name>  # Examples: ChatOpenAI, ChatGooglePalm
```

主要抽象：

```
AIMessage, BaseMessage, HumanMessage
```

### BaseChatModel Class


```python
# BaseChatModel 是一个抽象类（ABC）
# [BaseMessageChunk] 表示：输入输出都是「消息」，支持流式输出
class BaseChatModel(BaseLanguageModel[BaseMessageChunk], ABC):
    cache: Optional[bool] = None
    """是否缓存响应。"""
    verbose: bool = Field(default_factory=_get_verbosity)
    """是否打印响应文本。"""
    callbacks: Callbacks = Field(default=None, exclude=True)
    """添加到运行追踪的回调函数。"""
    callback_manager: Optional[BaseCallbackManager] = Field(default=None, exclude=True)
    """添加到运行追踪的回调函数管理器。"""
    tags: Optional[List[str]] = Field(default=None, exclude=True)
    """添加到运行追踪的标签。"""
    metadata: Optional[Dict[str, Any]] = Field(default=None, exclude=True)
    """添加到运行追踪的元数据。"""

    # 需要子类实现的 _generate 抽象方法，必须。与LLMs 区别点。
    @abstractmethod
    def _generate(
        self,
        messages: List[BaseMessage], # 👈 核心！输入是消息列表
        stop: Optional[List[str]] = None,
        run_manager: Optional[CallbackManagerForLLMRun] = None,
        **kwargs: Any,
    ) -> ChatResult:  # 👈 输出是 ChatResult（消息格式）

```

### ChatOpenAI Class（调用 Chat Completion API）


```python
class ChatOpenAI(BaseChatModel):
    """OpenAI Chat大语言模型的包装器。

    要使用，您应该已经安装了``openai`` python包，并且
    环境变量``OPENAI_API_KEY``已使用您的API密钥进行设置。

    即使未在此类上明确保存，也可以传入任何有效的参数
    至openai.create调用。
    """

    @property
    def lc_secrets(self) -> Dict[str, str]:
        return {"openai_api_key": "OPENAI_API_KEY"} # 自动从环境变量读取 API Key，不硬编码，更安全。

    @property
    def lc_serializable(self) -> bool:
        return True

    client: Any = None  #: :meta private:
    model_name: str = Field(default="gpt-3.5-turbo", alias="model")
    """要使用的模型名。"""
    temperature: float = 0.7
    """使用的采样温度。"""
    model_kwargs: Dict[str, Any] = Field(default_factory=dict)
    """保存任何未明确指定的`create`调用的有效模型参数。"""
    openai_api_key: Optional[str] = None
    """API请求的基础URL路径，
    如果不使用代理或服务仿真器，请留空。"""
    openai_api_base: Optional[str] = None
    openai_organization: Optional[str] = None
    # 支持OpenAI的显式代理
    openai_proxy: Optional[str] = None
    request_timeout: Optional[Union[float, Tuple[float, float]]] = None
    """请求OpenAI完成API的超时。默认为600秒。"""
    max_retries: int = 6
    """生成时尝试的最大次数。"""
    streaming: bool = False
    """是否流式传输结果。"""
    n: int = 1
    """为每个提示生成的聊天完成数。"""
    max_tokens: Optional[int] = None
    """生成的最大令牌数。"""
    tiktoken_model_name: Optional[str] = None
    """使用此类时传递给tiktoken的模型名称。
    Tiktoken用于计算文档中的令牌数以限制
    它们在某个限制之下。默认情况下，当设置为None时，这将
    与嵌入模型名称相同。但是，在某些情况下，
    您可能希望使用此嵌入类，模型名称不
    由tiktoken支持。这可能包括使用Azure嵌入或
    使用其中之一的多个模型提供商公开类似OpenAI的
    API但模型不同。在这些情况下，为了避免在调用tiktoken时出错，
    您可以在这里指定要使用的模型名称。"""


```

In [7]:
from langchain_community.chat_models import ChatZhipuAI
import os
from dotenv import load_dotenv
load_dotenv()
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
import time
# ======================
# 1. 初始化智谱聊天模型
# ======================

llm = ChatZhipuAI(model="glm-4.5-air", api_key=os.getenv("ZHIPUAI_API_KEY"),streaming=True)

# ==========================================
# 功能1：基础消息调用（System + User 消息）
# ==========================================
messages = [
    SystemMessage(content="你是一个专业的编程助手"),
    HumanMessage(content="什么是LangChain？"),
]

response = llm.invoke(messages)
print("AI 回答：", response.content)

# ==========================================
# 功能2：流式输出（打字机效果）
# ==========================================

for chunk in llm.stream(messages):
    print(chunk.content, end="", flush=True)
    time.sleep(0.05)
print()



AI 回答： LangChain 是一个**开源框架**，专门用于**构建基于大型语言模型（LLM）的应用程序**。它的核心目标是**简化开发过程**，让开发者能够更轻松地**将强大的 LLM 能力（如理解、生成、推理）与外部数据源（如数据库、API、文档）以及计算资源（如代码执行）连接起来**，从而创建更智能、更强大的应用。

你可以把 LangChain 想象成一个**“胶水”或“工具箱”**，它提供了组件和接口，帮助你将 LLM 的核心能力与外部世界连接起来，构建复杂的、有状态的应用。

## LangChain 的核心概念和组件

1.  **模型：**
    *   提供与各种 LLM 提供商（如 OpenAI, Anthropic, Cohere, Hugging Face, Google Gemini 等）的接口。
    *   支持不同的模型类型：聊天模型（生成对话）、文本生成模型（生成文本）。
    *   提供模型选择和调用的统一方式。

2.  **提示：**
    *   **提示模板：** 这是 LangChain 最核心的概念之一。它允许你**动态生成发送给 LLM 的提示**。你可以将变量（如用户问题、文档内容）插入到预定义的提示模板中，形成完整的提示。这使得提示可以适应不同的输入和上下文。
    *   **示例选择器：** 在提示中包含相关的示例（如 Few-shot learning）。
    *   **输出解析器：** 将 LLM 返回的原始文本（通常是字符串）结构化为你需要的格式（如 JSON, 对象）。

3.  **记忆：**
    *   **关键挑战：** 标准 LLM 是**无状态**的，每次调用都是独立的，无法记住之前的对话或信息。
    *   **解决方案：** LangChain 提供了各种**记忆组件**，用于在多个 LLM 调用之间**存储和检索信息**（如对话历史、用户偏好）。
    *   **类型：** 对话缓冲（ConversationBuffer）、对话摘要（ConversationSummary）、基于实体（EntityMemory）等。

4.  **链：**
    *   **核心抽象：** 链是将**多个组件（模型、提示、工具、其他链）按特定顺序组合**起来执行一个